# Build and push with Kaniko (Tekton-equivalent)

Mirrors `task-build-push.yaml` from the fixer-agent Tekton Task:

1. **Registry auth** — Read the pod ServiceAccount token and write `/tmp/kaniko-docker/config.json` with `openshift:<token>` (base64) for `REGISTRY_HOST`, matching the internal registry key used by Docker/Kaniko.
2. **Build and push** — Run Kaniko executor: `--dockerfile`, `--context=dir://<repo>`, `--destination=<IMAGE>`, `--skip-tls-verify`.

**Non-privileged:** no Buildah/Podman; same pattern as Tekton.

**Elyra runtime image:** must include **Python** (for notebook execution) and the **Kaniko** executor binary (see `pipelines/elyra-kaniko-runtime.Containerfile`). Set `REPLACE_ME_KANIKO_ELYRA_RUNTIME_IMAGE` on this pipeline node.

**Cluster:** grant the pipeline step ServiceAccount **`system:image-pusher`** on the target namespace (same as Tekton PipelineRun SA).

**Environment variables**

| Variable | Default | Meaning |
|----------|---------|--------|
| `IMAGE` | (required) | Full image reference, e.g. `image-registry.openshift-image-registry.svc:5000/ns/name:tag` |
| `REGISTRY_HOST` | `image-registry.openshift-image-registry.svc:5000` | Host:port used as the key in `config.json` `auths` (must match the registry in `IMAGE`). |
| `CONTAINERFILE` | `Containerfile` | Path to the Dockerfile **relative to the build context** (repo root). |
| `REPO_ROOT` | `cwd` | Repository root passed as Kaniko `--context`. |
| `KANIKO_EXECUTOR` | `/kaniko/executor` | Kaniko executor path. |


In [ ]:
import base64
import json
import os
import subprocess

ROOT = os.path.abspath(os.environ.get("REPO_ROOT", os.getcwd()))
IMAGE = os.environ.get("IMAGE", "").strip()
REGISTRY_HOST = os.environ.get(
    "REGISTRY_HOST", "image-registry.openshift-image-registry.svc:5000"
).strip()
DOCKERFILE_REL = os.environ.get("CONTAINERFILE", "Containerfile").strip()
KANIKO = os.environ.get("KANIKO_EXECUTOR", "/kaniko/executor")
DOCKER_CONFIG_DIR = os.environ.get("DOCKER_CONFIG", "/tmp/kaniko-docker")

if "REPLACE_ME_" in IMAGE or not IMAGE:
    raise RuntimeError(
        "Set IMAGE to the full destination (namespace/ImageStream:tag on the internal registry)."
    )

ctx_dockerfile = os.path.join(ROOT, DOCKERFILE_REL)
if not os.path.isfile(ctx_dockerfile):
    raise FileNotFoundError(f"Missing {ctx_dockerfile} (context={ROOT})")

weights = os.path.join(ROOT, "runs-openshift", "exp1", "weights", "best.pt")
if not os.path.isfile(weights):
    raise FileNotFoundError(
        f"Missing {weights}. Run the training notebook step first in the same workspace."
    )

token_file = "/var/run/secrets/kubernetes.io/serviceaccount/token"
if not os.path.isfile(token_file) or not os.access(token_file, os.R_OK):
    raise FileNotFoundError(
        f"ServiceAccount token not available at {token_file}. "
        "Run this pipeline on OpenShift/Kubernetes with automount SA token enabled."
    )

with open(token_file, encoding="utf-8") as f:
    token = "".join(f.read().split())

auth_b64 = base64.b64encode(f"openshift:{token}".encode()).decode("ascii")
cfg = {"auths": {REGISTRY_HOST: {"auth": auth_b64}}}
os.makedirs(DOCKER_CONFIG_DIR, exist_ok=True)
cfg_path = os.path.join(DOCKER_CONFIG_DIR, "config.json")
with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(cfg, f)
    f.write("\n")
print("Wrote", cfg_path, "for registry", REGISTRY_HOST, flush=True)

if not os.path.isfile(KANIKO):
    raise FileNotFoundError(
        f"Kaniko executor not found at {KANIKO}. Use a pipeline runtime image that includes "
        "/kaniko/executor (see pipelines/elyra-kaniko-runtime.Containerfile)."
    )

args = [
    KANIKO,
    f"--dockerfile={DOCKERFILE_REL}",
    f"--context=dir://{ROOT}",
    f"--destination={IMAGE}",
    "--skip-tls-verify",
]
print("Running:", " ".join(args), flush=True)
env = {**os.environ, "DOCKER_CONFIG": DOCKER_CONFIG_DIR}
subprocess.run(args, check=True, env=env)
print("Pushed:", IMAGE, flush=True)
